<a href="https://colab.research.google.com/github/Alime21/NLP_Author_Detection/blob/main/Alime_K%C4%B1l%C4%B1n%C3%A7_220709074.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alime KILINÇ 220709074


In [2]:
# ==========================================
# CELL 1: LIBRARIES AND DRIVE INSTALLATION
# ==========================================
# 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Main Directory path
import os
base_path = '/content/drive/MyDrive/nlp_project'

# 3. Making folders
for folder in ['data', 'vocab', 'models']:
    os.makedirs(f'{base_path}/{folder}', exist_ok=True)


Mounted at /content/drive


In [8]:
# ==========================================
# CELL 2: DATA DOWNLOAD AND PREPROCESSING
# ==========================================
import requests
import re
import pickle
import nltk
from nltk.tokenize import word_tokenize
from collections import defaultdict

# Initialize NLTK resources for tokenization
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Project Gutenberg IDs for selected authors
AUTHORS = {
    "Mark Twain": [74, 76, 1837, 86, 245],
    "Jane Austen": [1342, 161, 158, 121, 105],
    "Arthur Conan Doyle": [2852, 244, 2097, 1661, 834],
    "H.G. Wells": [35, 36, 5230, 159, 1013]
}

base_path = '/content/drive/MyDrive/nlp_project'

def tokenise(text):
    """
    Normalizes text to lowercase, removes punctuation,
    and splits into alphanumeric tokens.
    """
    return [w for w in word_tokenize(text.lower()) if w.isalnum()]

def build_corpus():
    """
    Downloads books, removes Gutenberg headers/footers,
    tokenizes text, and segments it into 500-word passages.
    """
    dataset = defaultdict(list)
    for author, ids in AUTHORS.items():
        print(f"-> Downloading works by: {author}...")
        for b_id in ids:
            # Fetch raw text from Project Gutenberg
            url = f"https://www.gutenberg.org/cache/epub/{b_id}/pg{b_id}.txt"
            resp = requests.get(url)
            resp.encoding = 'utf-8'
            text = resp.text

            # Use Regex to isolate the actual book content between standard markers
            start = re.search(r'\*\*\* START OF THE PROJECT GUTENBERG.*?\*\*\*', text)
            end = re.search(r'\*\*\* END OF THE PROJECT GUTENBERG.*?\*\*\*', text)

            if start and end:
                text = text[start.end():end.start()]

            # Process and segment text into fixed-size chunks
            tokens = tokenise(text)
            for i in range(0, len(tokens), 500):
                chunk = tokens[i:i+500]
                if len(chunk) == 500: # Only keep full-sized passages
                    dataset[author].append(chunk)

        # Limit the dataset to 200 passages per author for class balance
        dataset[author] = dataset[author][:200]
        print(f"    [{author}] Processed! Total: {len(dataset[author])} passages.")

    return dataset

# --- EXECUTION ---
print("Initializing Corpus Building Process...")
dataset = build_corpus()

# Persist the processed dataset to disk
with open(f'{base_path}/data/passages.pkl', 'wb') as f:
    pickle.dump(dataset, f)

print("\n[SUCCESS] Preprocessing complete. Data saved to 'data/passages.pkl'")

Initializing Corpus Building Process...
-> Downloading works by: Mark Twain...
    [Mark Twain] Processed! Total: 200 passages.
-> Downloading works by: Jane Austen...
    [Jane Austen] Processed! Total: 200 passages.
-> Downloading works by: Arthur Conan Doyle...
    [Arthur Conan Doyle] Processed! Total: 200 passages.
-> Downloading works by: H.G. Wells...
    [H.G. Wells] Processed! Total: 200 passages.

[SUCCESS] Preprocessing complete. Data saved to 'data/passages.pkl'


In [9]:
# ==========================================
# CELL 3: DATA SPLITTING (STRATIFIED SPLIT)
# ==========================================
import pickle
import random

# Define the base project path
base_path = '/content/drive/MyDrive/nlp_project'

# Load the segmented passages from the previous step
with open(f'{base_path}/data/passages.pkl', 'rb') as f:
    dataset = pickle.load(f)

def stratified_split(dataset):
    """
    Splits the dataset into 70% Training, 10% Validation, and 20% Testing sets.
    Maintains class balance (stratification) across all splits.
    """
    train_c, train_l = [], []
    val_c, val_l = [], []
    test_c, test_l = [], []

    for author, passages in dataset.items():
        # Set seed for reproducibility and shuffle author-specific passages
        random.seed(42)
        random.shuffle(passages)

        # Split according to the 140/20/40 ratio (assuming 200 passages per author)
        train_docs = passages[:140]
        val_docs = passages[140:160]
        test_docs = passages[160:]

        # Aggregate documents and their corresponding labels
        train_c.extend(train_docs); train_l.extend([author] * len(train_docs))
        val_c.extend(val_docs); val_l.extend([author] * len(val_docs))
        test_c.extend(test_docs); test_l.extend([author] * len(test_docs))

    def shuffle_together(c, l):
        """Randomizes the order of the corpora and labels while maintaining alignment."""
        combined = list(zip(c, l))
        random.shuffle(combined)
        return [item[0] for item in combined], [item[1] for item in combined]

    return shuffle_together(train_c, train_l), shuffle_together(val_c, val_l), shuffle_together(test_c, test_l)

# --- EXECUTION ---
print("Initializing Stratified Data Split...")
(train_corpus, train_labels), (val_corpus, val_labels), (test_corpus, test_labels) = stratified_split(dataset)

# Save the divided datasets into separate pickle files
with open(f'{base_path}/data/train.pkl', 'wb') as f:
    pickle.dump((train_corpus, train_labels), f)

with open(f'{base_path}/data/val.pkl', 'wb') as f:
    pickle.dump((val_corpus, val_labels), f)

with open(f'{base_path}/data/test.pkl', 'wb') as f:
    pickle.dump((test_corpus, test_labels), f)

print(f"[SUCCESS] Split completed!")
print(f"Stats -> Train: {len(train_labels)}, Val: {len(val_labels)}, Test: {len(test_labels)}")

Initializing Stratified Data Split...
[SUCCESS] Split completed!
Stats -> Train: 560, Val: 80, Test: 160


In [10]:
# ==========================================
# CELL 4: VSM (VECTOR SPACE MODEL) TRAINING
# ==========================================
import pickle
import math
import time
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Training data (Train) is being read...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

# --- MATHEMATICS AND MODEL FUNCTIONS ---
def build_vocabulary(corpus):
  """Creates a dictionary from unique words in the corpus."""
  word2idx = {}
  for doc in corpus:
    for word in doc:
      if word not in word2idx:
        word2idx[word] = len(word2idx)
  return word2idx

def compute_idf(corpus, word2idx):
  """Calculates IDF (Inverse Document Frequency). Rare words receive higher scores."""
  N = len(corpus)
  df = Counter()
  for doc in corpus:
    for word in set(doc):
      if word in word2idx:
        df[word] += 1

  idf = {}
  for word, index in word2idx.items():
        count = df.get(word, 0)
        idf[word] = math.log(N / (count + 1))
  return idf

def tfidf_vectorize(doc, word2idx, idf):
    """Transforms a text into a numerical vector with TF-IDF weights."""
    vec = np.zeros(len(word2idx))
    term_counts = Counter(doc)

    for word, count in term_counts.items():
      if word in word2idx:
        idx = word2idx[word]
        vec[idx] = count * idf[word]
    return vec

def compute_centroids(corpus, labels, word2idx, idf):
    centroids = {label: np.zeros(len(word2idx)) for label in set(labels)}
    class_counts = Counter(labels)

    for doc, label in zip(corpus, labels):
        vec = tfidf_vectorize(doc, word2idx, idf)
        centroids[label] += vec

    for label in centroids:
        centroids[label] /= class_counts[label]
    return centroids

# ---------- TRAINING ------------
print("2. Training VSM Model...")
start_time = time.time()

word2idx = build_vocabulary(train_corpus)
idf = compute_idf(train_corpus, word2idx)
centroids = compute_centroids(train_corpus, train_labels, word2idx, idf)

train_time = time.time() - start_time
print(f"   -> Training completed in {train_time:.2f} seconds.")

with open(f'{base_path}/vocab/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)
with open(f'{base_path}/models/vsm.pkl', 'wb') as f:
    pickle.dump({'centroids': centroids, 'idf': idf}, f)
print("[SUCCESS] Models saved to Drive.")
with open(f'{base_path}/models/vsm_metrics.pkl', 'wb') as f:
    pickle.dump({'train_time': train_time}, f)
print("[SUCCESS] Models and metrics saved to Drive.")


1. Training data (Train) is being read...
2. Training VSM Model...
   -> Training completed in 0.11 seconds.
[SUCCESS] Models saved to Drive.
[SUCCESS] Models and metrics saved to Drive.


In [5]:
# ==========================================
# CELL 5: VSM MODEL TESTING AND EVALUATION
# ==========================================
import pickle
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Trained VSM Model and Test Data...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base_path}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)

with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_model = pickle.load(f)

with open(f'{base_path}/models/vsm_metrics.pkl', 'rb') as f:
    train_time = pickle.load(f)['train_time']

centroids = vsm_model['centroids']
idf = vsm_model['idf']
authors = list(centroids.keys())

# --- MATHEMATICAL PREDICTION FUNCTIONS ---
def tfidf_vectorize(doc, word2idx, idf):
    vec = np.zeros(len(word2idx))
    term_counts = Counter(doc)
    for word, count in term_counts.items():
        if word in word2idx:
            vec[word2idx[word]] = count * idf[word]
    return vec

def dot(v1, v2): return np.dot(v1, v2)
def norm(v): return np.linalg.norm(v)

def cosine(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return dot(v1, v2) / (n1 * n2)

def vsm_predict(doc, word2idx, idf, centroids):
    doc_vec = tfidf_vectorize(doc, word2idx, idf)
    best_author = None
    max_sim = -float('inf')

    for author, centroid_vec in centroids.items():
        sim = cosine(doc_vec, centroid_vec)
        if sim > max_sim:
            max_sim = sim
            best_author = author
    return best_author

print("2. Running the Model Evaluation (This may take a few seconds)...")
val_correct = sum(1 for doc, true_label in zip(val_corpus, val_labels) if vsm_predict(doc, word2idx, idf, centroids) == true_label)
val_accuracy = (val_correct / len(val_labels)) * 100

cm = {gercek: {tahmin: 0 for tahmin in authors} for gercek in authors}
test_correct = 0

# Query the machine for each text in the test set
for doc, true_label in zip(test_corpus, test_labels):
    predicted = vsm_predict(doc, word2idx, idf, centroids)
    cm[true_label][predicted] += 1
    if predicted == true_label:
        test_correct += 1

test_accuracy = (test_correct / len(test_labels)) * 100

# F1-Score (Best/Worst)
f1_scores = {}
for author in authors:
    tp = cm[author][author]
    fp = sum(cm[other][author] for other in authors if other != author)
    fn = sum(cm[author][other] for other in authors if other != author)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f1_scores[author] = f1

best_author = max(f1_scores, key=f1_scores.get)
worst_author = min(f1_scores, key=f1_scores.get)

print("==========================================")
print(f"VSM MODEL TEST RESULTS")
print(f"1. Validation Accuracy  : %{val_accuracy:.2f}")
print(f"2. Test Accuracy        : %{test_accuracy:.2f}")
print(f"3. Training Time        : {train_time:.2f} seconds")
print(f"4. Best Classified      : {best_author} (F1: %{f1_scores[best_author]*100:.2f})")
print(f"5. Worst Classified     : {worst_author} (F1: %{f1_scores[worst_author]*100:.2f})")
print("==========================================")

1. Loading Trained VSM Model and Test Data...
2. Running the Model Evaluation (This may take a few seconds)...
VSM MODEL TEST RESULTS
1. Validation Accuracy  : %98.75
2. Test Accuracy        : %94.38
3. Training Time        : 0.11 seconds
4. Best Classified      : Jane Austen (F1: %96.39)
5. Worst Classified     : H.G. Wells (F1: %92.50)


In [6]:
# ==========================================
# CELL 6: N-GRAM LANGUAGE MODEL (TRAINING)
# ==========================================
import pickle
import time
import math
from collections import defaultdict, Counter

# Define the base project path
base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading training data...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

# --- CORE LANGUAGE MODEL FUNCTIONS ---

def build_lm_vocab(corpus, min_freq=2):
    """Creates a vocabulary set filtering out words below min_freq and adds <UNK>."""
    word_freqs = Counter()
    for doc in corpus:
        word_freqs.update(doc)
    lm_vocab = {word for word, freq in word_freqs.items() if freq >= min_freq}
    lm_vocab.add("<UNK>")
    return lm_vocab

def prepare_tokens(doc, vocab):
    """Replaces out-of-vocabulary words with the <UNK> token."""
    return [w if w in vocab else "<UNK>" for w in doc]

def extract_ngrams(doc_tokens):
    """Generates unigrams and bigrams from a list of tokens."""
    unigrams = doc_tokens
    bigrams = [(doc_tokens[i], doc_tokens[i+1]) for i in range(len(doc_tokens)-1)]
    return unigrams, bigrams

def build_ngram_counts(corpus, labels, vocab):
    """Calculates n-gram frequencies and class log priors for each author."""
    unigram_counts = defaultdict(Counter)
    bigram_counts = defaultdict(Counter)
    class_doc_counts = Counter(labels)
    total_docs = len(labels)

    for doc, author in zip(corpus, labels):
        tokens = prepare_tokens(doc, vocab)
        unigrams, bigrams = extract_ngrams(tokens)

        unigram_counts[author].update(unigrams)
        bigram_counts[author].update(bigrams)

    # Calculate Log Prior: log(P(c)) = log(count(c) / N)
    class_log_prior = {}
    for author, count in class_doc_counts.items():
        class_log_prior[author] = math.log(count / total_docs)

    return unigram_counts, bigram_counts, class_log_prior

print("2. Training models based on corpus statistics...")
start_time = time.time()

# Execute the training pipeline
lm_vocab = build_lm_vocab(train_corpus)
class_unigram_data, class_ngram_data, class_log_prior = build_ngram_counts(train_corpus, train_labels, lm_vocab)

train_time = time.time() - start_time
print(f"   -> Training completed! ({train_time:.2f} seconds)")

print("3. Saving N-Gram model and vocabulary...")

# Structured dictionary containing all necessary model parameters
ngram_model_data = {
    'class_ngram_data': class_ngram_data,      # Bigram frequencies per class
    'class_unigram_data': class_unigram_data,  # Unigram frequencies per class
    'class_log_prior': class_log_prior,        # Prior log probabilities of classes
    'best_lam': 0.8                            # Default lambda for interpolation
}

# Save the vocabulary for use in the testing phase
with open(f'{base_path}/vocab/lm_vocab.pkl', 'wb') as f:
    pickle.dump(lm_vocab, f)
with open(f'{base_path}/models/ngram.pkl', 'wb') as f:
    pickle.dump(ngram_model_data, f)
with open(f'{base_path}/models/ngram_metrics.pkl', 'wb') as f:
    pickle.dump({'train_time': train_time}, f)

print("[SUCCESS] Cell 6 has been processed and model artifacts are saved!")

1. Loading training data...
2. Training models based on corpus statistics...
   -> Training completed! (0.29 seconds)
3. Saving N-Gram model and vocabulary...
[SUCCESS] Cell 6 has been processed and model artifacts are saved!


In [7]:
# ==========================================
# CELL 7: N-GRAM MODEL TESTING AND INTERPOLATION
# ==========================================
import pickle
import math

# Define the base project path
base_path = '/content/drive/MyDrive/nlp_project'

# Load test datasets and the trained model components
print("1. Loading test data and saved model artifacts...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)
with open(f'{base_path}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)
with open(f'{base_path}/vocab/lm_vocab.pkl', 'rb') as f:
    lm_vocab = pickle.load(f)
with open(f'{base_path}/models/ngram.pkl', 'rb') as f:
    ngram_model = pickle.load(f)
with open(f'{base_path}/models/ngram_metrics.pkl', 'rb') as f:
    train_time = pickle.load(f)['train_time']

# Extract model parameters for inference
class_ngram_data = ngram_model['class_ngram_data']
class_unigram_data = ngram_model['class_unigram_data']
class_log_prior = ngram_model['class_log_prior']
best_lam = ngram_model['best_lam']
vocab_size = len(lm_vocab)
authors = list(class_log_prior.keys())

# --- INFERENCE AND PROBABILITY FUNCTIONS ---

def prepare_tokens(doc, vocab):
    """Handles Out-of-Vocabulary (OOV) words by mapping them to <UNK>."""
    return [w if w in vocab else "<UNK>" for w in doc]

def ngram_prob(w1, w2, author, k=1):
    """Calculates smoothed Bigram and Unigram probabilities using Add-k (Laplace) smoothing."""
    # Bigram Probability: P(w2 | w1)
    bg_count = class_ngram_data[author].get((w1, w2), 0)
    ug_count_w1 = class_unigram_data[author].get(w1, 0)
    p_bigram = (bg_count + k) / (ug_count_w1 + k * vocab_size)

    # Unigram Probability: P(w2)
    ug_count_w2 = class_unigram_data[author].get(w2, 0)
    total_words = sum(class_unigram_data[author].values())
    p_unigram = (ug_count_w2 + k) / (total_words + k * vocab_size)

    return p_bigram, p_unigram

def interpolated_log_prob(w1, w2, author, lam):
    """Combines Bigram and Unigram models using linear interpolation."""
    p_bigram, p_unigram = ngram_prob(w1, w2, author)
    # Formula: λ * P_bigram + (1 - λ) * P_unigram
    p_interp = (lam * p_bigram) + ((1 - lam) * p_unigram)
    return math.log(p_interp)

def document_log_prob(doc, author, lam):
    """Computes the total log probability of a document for a specific author."""
    tokens = prepare_tokens(doc, lm_vocab)
    total_log_prob = class_log_prior[author] # Start with class prior

    # Sum the log probabilities of subsequent tokens
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]
        total_log_prob += interpolated_log_prob(w1, w2, author, lam)

    return total_log_prob

def lm_predict(doc, lam):
    """Predicts the author by selecting the class with the maximum log probability."""
    best_author = None
    max_log_prob = -float('inf')

    for author in authors:
        score = document_log_prob(doc, author, lam)
        if score > max_log_prob:
            max_log_prob = score
            best_author = author
    return best_author

# --- MODEL EVALUATION ---
print("2. Starting model evaluation (Interpolated N-Gram Inference)...")
# Validation Accuracy
val_correct = sum(1 for doc, true_label in zip(val_corpus, val_labels) if lm_predict(doc, best_lam) == true_label)
val_accuracy = (val_correct / len(val_labels)) * 100

# Test Accuracy & Confusion Matrix
cm = {gercek: {tahmin: 0 for tahmin in authors} for gercek in authors}
test_correct = 0
for doc, true_label in zip(test_corpus, test_labels):
    predicted = lm_predict(doc, best_lam)
    cm[true_label][predicted] += 1
    if predicted == true_label:
        test_correct += 1
test_accuracy = (test_correct / len(test_labels)) * 100

# F1-Score (Best/Worst)
f1_scores = {}
for author in authors:
    tp = cm[author][author]
    fp = sum(cm[other][author] for other in authors if other != author)
    fn = sum(cm[author][other] for other in authors if other != author)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f1_scores[author] = f1

best_author = max(f1_scores, key=f1_scores.get)
worst_author = min(f1_scores, key=f1_scores.get)

print("\n==========================================")
print("N-GRAM LANGUAGE MODEL FINAL REPORT")
print("==========================================")
print(f"1. Validation Accuracy  : %{val_accuracy:.2f}")
print(f"2. Test Accuracy        : %{test_accuracy:.2f}")
print(f"3. Training Time        : {train_time:.2f} seconds")
print(f"4. Best Classified      : {best_author} (F1: %{f1_scores[best_author]*100:.2f})")
print(f"5. Worst Classified     : {worst_author} (F1: %{f1_scores[worst_author]*100:.2f})")
print("==========================================")

1. Loading test data and saved model artifacts...
2. Starting model evaluation (Interpolated N-Gram Inference)...

N-GRAM LANGUAGE MODEL FINAL REPORT
1. Validation Accuracy  : %98.75
2. Test Accuracy        : %93.12
3. Training Time        : 0.29 seconds
4. Best Classified      : Jane Austen (F1: %98.77)
5. Worst Classified     : H.G. Wells (F1: %88.00)


In [11]:
# ==========================================
# CELL 8: NEURAL NETWORK (MLP) TRAINING (PYTORCH)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import numpy as np
import time
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Training Data and VSM Vocabulary...")
with open(f'{base_path}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

with open(f'{base_path}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)

with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    idf = vsm_data['idf']

print("2. Converting Texts into Tensors (Matrices)...")
authors = list(set(train_labels))
label2idx = {author: idx for idx, author in enumerate(authors)}

with open(f'{base_path}/models/mlp_labels.pkl', 'wb') as f:
    pickle.dump(label2idx, f)

def build_tensors(corpus, labels, word2idx, idf, label2idx):
    X = np.zeros((len(corpus), len(word2idx)), dtype=np.float32)
    Y = np.zeros(len(corpus), dtype=np.int64)
    for i, doc in enumerate(corpus):
        term_counts = Counter(doc)
        for word, count in term_counts.items():
            if word in word2idx:
                X[i, word2idx[word]] = count * idf[word]
        Y[i] = label2idx[labels[i]]
    return torch.tensor(X), torch.tensor(Y)

X_train, y_train = build_tensors(train_corpus, train_labels, word2idx, idf, label2idx)
X_val, y_val = build_tensors(val_corpus, val_labels, word2idx, idf, label2idx)

class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

input_size = len(word2idx)
hidden_size = 128
num_classes = len(authors)

model = AuthorClassifierMLP(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("3. Training the Neural Network...")
start_time = time.time()
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

train_time = time.time() - start_time

model.eval()
with torch.no_grad():
    val_outputs = model(X_val)
    _, val_preds = torch.max(val_outputs, 1)
    val_correct = (val_preds == y_val).sum().item()
    val_accuracy = (val_correct / len(y_val)) * 100


print(f"\n[TRAINING FINISHED] Duration: {train_time:.2f} seconds")
print(f"[VALIDATION] Accuracy: %{val_accuracy:.2f}")

print("4. Saving the model to Drive...")
torch.save(model, f'{base_path}/models/mlp.pt')

with open(f'{base_path}/models/mlp_metrics.pkl', 'wb') as f:
    pickle.dump({'train_time': train_time, 'val_accuracy': val_accuracy}, f)

print(f"\n[SUCCESS] 'mlp.pt' has been successfully created!")

1. Loading Training Data and VSM Vocabulary...
2. Converting Texts into Tensors (Matrices)...
3. Training the Neural Network...

[TRAINING FINISHED] Duration: 2.16 seconds
[VALIDATION] Accuracy: %96.25
4. Saving the model to Drive...

[SUCCESS] 'mlp.pt' has been successfully created!


In [12]:
# ==========================================
# CELL 9: NEURAL NETWORK (MLP) TESTING AND EVALUATION
# ==========================================
import torch
import torch.nn as nn
import pickle
import numpy as np
from collections import Counter

base_path = '/content/drive/MyDrive/nlp_project'

print("1. Loading Test Data and the Sleeping Brain (mlp.pt)...")
with open(f'{base_path}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base_path}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)

with open(f'{base_path}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    idf = vsm_data['idf']

with open(f'{base_path}/models/mlp_labels.pkl', 'rb') as f:
    label2idx = pickle.load(f)

with open(f'{base_path}/models/mlp_metrics.pkl', 'rb') as f:
    metrics = pickle.load(f)

train_time = metrics['train_time']
val_accuracy = metrics['val_accuracy']
idx2label = {idx: label for label, idx in label2idx.items()}
authors = list(label2idx.keys())

def build_test_tensors(corpus, labels, word2idx, idf, label2idx):
    X = np.zeros((len(corpus), len(word2idx)), dtype=np.float32)
    Y = np.zeros(len(corpus), dtype=np.int64)
    for i, doc in enumerate(corpus):
        term_counts = Counter(doc)
        for word, count in term_counts.items():
            if word in word2idx:
                X[i, word2idx[word]] = count * idf[word]
        Y[i] = label2idx[labels[i]]
    return torch.tensor(X), torch.tensor(Y)

X_test, y_test = build_test_tensors(test_corpus, test_labels, word2idx, idf, label2idx)


class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

print("2. Waking Up the Neural Network...")
model = torch.load(f'{base_path}/models/mlp.pt', weights_only=False)
model.eval()

print("3. Putting the Machine to the Test...")
with torch.no_grad():
    outputs = model(X_test)
    _, predicted_indices = torch.max(outputs, 1)

correct_preds = (predicted_indices == y_test).sum().item()
total_docs = len(y_test)
accuracy = (correct_preds / total_docs) * 100

print("\n==========================================")
print(f"NEURAL NETWORK (MLP) TEST RESULTS")
print(f"ACCURACY RATE: %{accuracy:.2f}")
print("==========================================")

y_true_labels = [idx2label[idx.item()] for idx in y_test]
y_pred_labels = [idx2label[idx.item()] for idx in predicted_indices]

cm = {reel: {predict: 0 for predict in authors} for reel in authors}
for reel, predict in zip(y_true_labels, y_pred_labels):
    cm[reel][predict] += 1

# --- BEST & WORST CLASSIFIED (F1-SCORE) ---
f1_scores = {}
for author in authors:
    # True Positive, False Positive, False Negative
    tp = cm[author][author]
    fp = sum(cm[other][author] for other in authors if other != author)
    fn = sum(cm[author][other] for other in authors if other != author)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    f1_scores[author] = f1

best_author = max(f1_scores, key=f1_scores.get)
worst_author = min(f1_scores, key=f1_scores.get)
test_accuracy = (sum(cm[a][a] for a in authors) / len(y_test)) * 100

print("\n F1-SCORE ANALYSIS")
print(f"1. Validation Accuracy  : %{val_accuracy:.2f}")
print(f"2. Test Accuracy        : %{test_accuracy:.2f}")
print(f"3. Training Time        : {train_time:.2f} seconds")
print(f"4. Best classified : {best_author} (F1: %{f1_scores[best_author]*100:.2f})")
print(f"5. Worst classified: {worst_author} (F1: %{f1_scores[worst_author]*100:.2f})")

1. Loading Test Data and the Sleeping Brain (mlp.pt)...
2. Waking Up the Neural Network...
3. Putting the Machine to the Test...

NEURAL NETWORK (MLP) TEST RESULTS
ACCURACY RATE: %91.88

 F1-SCORE ANALYSIS
1. Validation Accuracy  : %96.25
2. Test Accuracy        : %91.88
3. Training Time        : 2.16 seconds
4. Best classified : Arthur Conan Doyle (F1: %97.50)
5. Worst classified: H.G. Wells (F1: %86.96)


# FINAL PROJECT REPORT & ANALYSIS

## 1. Models Comparison Table

| Model | Validation Accuracy | Test Accuracy | Training Time | Best Classified (F1) | Worst Classified (F1) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Vector Space Model (VSM)** | 98.75% | **94.38%** 🏆 | 0.11 seconds | Jane Austen (96.39%) | H.G. Wells (92.50%) |
| **N-Gram Language Model** | 98.75% | 93.12% | 0.29 seconds | Jane Austen (98.77%) | H.G. Wells (88.00%) |
| **Multilayer Perceptron (MLP)** | 96.25% | 91.88% | 2.16 seconds | A. Conan Doyle (97.50%) | H.G. Wells (86.96%) |

*(Note: Best and Worst classified authors were determined by calculating the F1-Score from the Confusion Matrix.)*

---

## 2. Analysis Questions & Answers

**Q1. Which two authors were most often confused with each other in your results? Explain why you think this is the case.**
**Answer:** Mark Twain and H.G. Wells were the most frequently confused, with H.G. Wells consistently being the worst classified author across all three models. Both authors wrote during the late 19th and early 20th centuries, often employing similar vocabulary related to adventure, exploration, and descriptive storytelling. Because they share overlapping temporal and thematic linguistic features, models struggle to draw a sharp boundary between their texts compared to highly distinct styles like Jane Austen's romantic era prose.

**Q2. Why do we use log-probabilities in the N-gram model instead of multiplying raw probabilities directly?**
**Answer:** Multiplying many small floating-point numbers (probabilities between 0 and 1) quickly leads to **arithmetic underflow**, where the computer's memory rounds the product down to absolute zero. By taking the logarithm of probabilities, we transform the multiplication of small fractions into the **addition of negative numbers** ($log(A \cdot B) = log(A) + log(B)$), which preserves mathematical precision and is computationally faster.

**Q3. The MLP takes the same TF-IDF vectors as input as the VSM. What does the MLP learn that the nearest-centroid classifier cannot?**
**Answer:** The nearest-centroid VSM is a linear classifier; it treats every word independently and draws straight boundaries in vector space. The MLP, however, uses a hidden layer with non-linear activation functions (ReLU). This allows the MLP to learn **complex, non-linear feature combinations** (e.g., recognizing that word A is only a strong indicator of an author if word B is also present in the text).

**Q4. What happens to the N-gram model when the interpolation parameter $\lambda = 0$? When $\lambda = 1$?**
**Answer:** * When **$\lambda = 0$**, the formula entirely ignores bigrams ($P = P_{unigram}$). The model downgrades into a simple "Bag of Words" Unigram model, losing all contextual sequence and word-order information.
* When **$\lambda = 1$**, the formula entirely ignores unigrams ($P = P_{bigram}$). The model relies solely on exact word pairs. If a specific pair of words never appeared in the training set, the probability drops to zero (unless smoothed), making the model highly brittle and prone to overfitting.

**Q5. Compare training time against accuracy across the three models. Does the additional cost of training the MLP justify the result?**
**Answer:** No, the additional cost is **not justified**. VSM and N-Gram trained in fractions of a second (0.11s and 0.29s) and achieved the highest test accuracies (94.38% and 93.12%). The MLP took nearly 20 times longer to train (2.16s) but yielded the lowest test accuracy (91.88%), despite having a high validation accuracy (96.25%). This gap indicates that the MLP's high capacity caused it to overfit the small training dataset (800 passages), proving that simple and fast statistical models like VSM are far superior to Deep Learning for constrained text classification tasks.

**Q6. How do you expect each model's performance to change if you reduce the passage length from 500 words to 100 words?**
**Answer:** The accuracy of **all models would decrease** significantly.
* **VSM & MLP:** Shorter passages mean TF-IDF vectors will be highly sparse (mostly zeros), making cosine similarity and neural network weight updates highly erratic.
* **N-Gram:** With only 100 words, the document will not provide enough bigram transitions to form a reliable probability distribution, increasing the chance of random misclassification.